# **Fine-Tuning BERT on a Kaggle Dataset**


Fine-tuning BERT means taking a pre-trained BERT model and training it further on a specific dataset (like Kaggle) to perform a particular task such as sentiment analysis or text classification.

First, the dataset is cleaned and preprocessed. Then, the text is tokenized using the BERT tokenizer to convert it into a format the model can understand. After that, the pre-trained BERT model is loaded and trained (fine-tuned) on the dataset using an optimizer like AdamW with a small learning rate.

Finally, the model is evaluated using metrics such as accuracy, precision, recall, and F1-score to measure its performance.

## **1.Data Preprocessing**

Removed missing values using dropna().

Converted text to lowercase.

Removed URLs, special characters, and extra spaces.

Cleaned text for better model performance.

In [ ]:
!pip install datasets

from datasets import load_dataset
import pandas as pd
import re

# load IMDB dataset directly
dataset = load_dataset("imdb")

# convert to pandas
df = pd.DataFrame(dataset['train'])

# handle missing values
df = df.dropna()

# clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

df.head()

# **2.Data Splitting**

Split data into train, validation, and test sets.

Helps in training and evaluating model performance.

Train → learn, Test → check

In [ ]:
from sklearn.model_selection import train_test_split

# first split (train + temp)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.3, random_state=42
)

# second split (validation + test)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

# **3. Tokenization**

Used bert-base-uncased tokenizer.

Converted text into tokens for BERT model.

Text → numbers (tokens).


In [ ]:
from transformers import BertTokenizer

# load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# tokenize data
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)

# **4. Model Building**

Loaded pre-trained BERT model.

Used for text classification.

Load model for prediction.


In [ ]:
from transformers import AutoModelForSequenceClassification

# load pre-trained BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2   # binary classification
)

# **5. Fine-Tuning**


Trained model using AdamW optimizer.

Learning rate set to 2e-5.

Model learns from data.

In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# create datasets
train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

# **6.Model Evaluation**

Evaluated using Accuracy, Precision, Recall, F1 Score

Used confusion matrix for analysis

Check performance

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

# unfreeze last 2 layers
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True